In [ ]:
import math
import os
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import wandb
from datasets import DatasetDict, load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from trl import SFTConfig, SFTTrainer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
load_dotenv()

login(os.getenv("HF_TOKEN"))
wandb.login()
%env WANDB_PROJECT=finetuning-trial

SEED = 42
TRAIN_SIZE = 16384
TEST_SIZE = 1024
VALIDATION_SIZE = 1024
MAX_NEW_TOKENS = 512

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
NEW_MODEL_NAME = "finetuning-trial_lr2e4_r16_" + MODEL_NAME.replace("/", "_")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "finetuning_trial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
dataset_complete = load_dataset(
    "leandrodevai/medical-meadow-medical-flashcards-splitted"
)


In [3]:
dataset_small = DatasetDict(
    {
        "train": dataset_complete["train"]
        .shuffle(seed=SEED)
        .select(range(min(TRAIN_SIZE, len(dataset_complete["train"])))),
        "evaluation": dataset_complete["evaluation"]
        .shuffle(seed=SEED)
        .select(range(min(VALIDATION_SIZE, len(dataset_complete["evaluation"])))),
        "test": dataset_complete["test"]
        .shuffle(seed=SEED)
        .select(range(min(TEST_SIZE, len(dataset_complete["test"])))),
    }
)


SYSTEM_PROMPT = "You are a expert on medical subjects.\n"


def preprocess_function(example):
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT + example["instruction"],
            },
            {
                "role": "user",
                "content": example["input"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["output"],
            }
        ],
    }


dataset_small = dataset_small.map(
    preprocess_function, remove_columns=dataset_complete["train"].column_names
)


In [ ]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    dtype = torch.float32
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16
    )

    bnb_4bit_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=dtype,
        quantization_config=bnb_4bit_config,
        device_map=DEVICE,
    )

    model = prepare_model_for_kbit_training(model)
    return model, tokenizer


model, tokenizer = load_model(MODEL_NAME)


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
    # target_modules=[
    #     "q_proj",
    #     "k_proj",
    #     "v_proj",
    #     "o_proj",
    # ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

early_stopping_callback = [EarlyStoppingCallback(early_stopping_patience=10)]

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    completion_only_loss=True,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=32,
    save_steps=32,
    optim="paged_adamw_8bit",
    packing=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to=["wandb"] if os.getenv("WANDB_API_KEY") else [],
    load_best_model_at_end=True,
    run_name=NEW_MODEL_NAME,
)


def config_value(value):
    return value.value if hasattr(value, "value") else value


trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset_small["train"],
    eval_dataset=dataset_small["evaluation"],
    processing_class=tokenizer,
    callbacks=early_stopping_callback,
)


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [ ]:
trainer.train()

In [7]:
ANSWER_PREFIX_RE = re.compile(r"^\s*(?:answer|assistant)\s*:\s*", flags=re.IGNORECASE)
ARTICLE_RE = re.compile(r"\b(a|an|the)\b", flags=re.IGNORECASE)
PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)


def clean_answer(text):
    text = (text or "").strip()
    return ANSWER_PREFIX_RE.sub("", text).strip()


def normalize_answer(text):
    text = (text or "").lower()
    text = ARTICLE_RE.sub(" ", text)
    text = PUNCT_RE.sub(" ", text)
    return " ".join(text.split())


def answer_tokens(text):
    normalized = normalize_answer(text)
    return normalized.split() if normalized else []


def token_f1(prediction, reference):
    prediction_tokens = answer_tokens(prediction)
    reference_tokens = answer_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0

    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def get_reference(example):
    completion = example["completion"][0]["content"]
    return {"reference_answer": clean_answer(completion)}


def build_scoring_prompt(tokenizer, example):
    return tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,
        add_special_tokens=False,
        enable_thinking=False,
    )


def cuda_device_ids():
    return list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []


def synchronize_cuda():
    for device_id in cuda_device_ids():
        torch.cuda.synchronize(device_id)


def reset_cuda_peak_memory():
    if not torch.cuda.is_available():
        return
    synchronize_cuda()
    for device_id in cuda_device_ids():
        torch.cuda.reset_peak_memory_stats(device_id)


def cuda_peak_memory_mb():
    if not torch.cuda.is_available():
        return {
            "vram_peak_allocated_mb": float("nan"),
            "vram_peak_reserved_mb": float("nan"),
        }
    synchronize_cuda()
    allocated = sum(torch.cuda.max_memory_allocated(i) for i in cuda_device_ids())
    reserved = sum(torch.cuda.max_memory_reserved(i) for i in cuda_device_ids())
    return {
        "vram_peak_allocated_mb": allocated / 1024**2,
        "vram_peak_reserved_mb": reserved / 1024**2,
    }


In [8]:
@torch.inference_mode()
def generate_batch(model, tokenizer, examples):
    prompts = [build_scoring_prompt(tokenizer, example) for example in examples]
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, add_special_tokens=True
    ).to(model.device)

    synchronize_cuda()
    start_time = time.perf_counter()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    synchronize_cuda()
    inference_seconds = time.perf_counter() - start_time

    new_tokens = output_ids[:, inputs["input_ids"].shape[1] :]
    generated_token_count = 0
    for row in new_tokens:
        eos_positions = (
            (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
            if tokenizer.eos_token_id is not None
            else []
        )
        if len(eos_positions) > 0:
            token_count = int(eos_positions[0].item()) + 1
        elif tokenizer.pad_token_id is not None:
            token_count = int(row.ne(tokenizer.pad_token_id).sum().item())
        else:
            token_count = row.shape[0]
        generated_token_count += token_count

    stats = {
        "generated_tokens": generated_token_count,
        "inference_seconds": inference_seconds,
        "tokens_per_second": generated_token_count / inference_seconds
        if inference_seconds > 0
        else float("nan"),
    }
    generations = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return generations, stats


In [9]:
BATCH_SIZE = 16


def reference_answer_logprobs(model, tokenizer, example):
    prompt = build_scoring_prompt(tokenizer, example)
    reference_answer = get_reference(example)["reference_answer"]
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=True
    ).input_ids[0]
    target_ids = tokenizer(
        reference_answer, return_tensors="pt", add_special_tokens=False
    ).input_ids[0]

    if target_ids.numel() == 0:
        return {
            "reference_answer_logprob_sum": float("nan"),
            "reference_answer_nll_mean": float("nan"),
            "reference_answer_token_perplexity": float("nan"),
            "reference_answer_tokens": 0,
        }

    input_ids = torch.cat([prompt_ids, target_ids]).unsqueeze(0).to(model.device)
    attention_mask = torch.ones_like(input_ids).to(model.device)
    with torch.inference_mode():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    prompt_length = prompt_ids.shape[0]
    target_logits = logits[
        0, prompt_length - 1 : prompt_length - 1 + target_ids.shape[0], :
    ]
    target_ids = target_ids.to(model.device)
    log_probs = torch.nn.functional.log_softmax(target_logits, dim=-1)
    token_logprobs = log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
    logprob_sum = float(token_logprobs.sum().item())
    nll_mean = float(-token_logprobs.mean().item())

    return {
        "reference_answer_logprob_sum": logprob_sum,
        "reference_answer_nll_mean": nll_mean,
        "reference_answer_token_perplexity": math.exp(nll_mean)
        if math.isfinite(nll_mean)
        else float("nan"),
        "reference_answer_tokens": int(target_ids.shape[0]),
    }


def score_reference_answers(model, tokenizer, eval_dataset):
    rows = []
    synchronize_cuda()
    start_time = time.perf_counter()
    for index, example in enumerate(
        tqdm(eval_dataset, desc="reference logprob", leave=False)
    ):
        rows.append(
            {
                "example_index": index,
                **reference_answer_logprobs(model, tokenizer, example),
            }
        )
    synchronize_cuda()
    return pd.DataFrame(rows), time.perf_counter() - start_time


def generate_model_predictions(model_id, eval_dataset):
    rows = []
    batch_stats = []
    reference_logprob_seconds = float("nan")
    reset_cuda_peak_memory()
    try:
        for start in tqdm(range(0, len(eval_dataset), BATCH_SIZE), desc=model_id):
            examples = [
                eval_dataset[i]
                for i in range(start, min(start + BATCH_SIZE, len(eval_dataset)))
            ]
            generations, stats = generate_batch(model, tokenizer, examples)
            batch_stats.append(stats)
            for index, (example, generation) in enumerate(
                zip(examples, generations), start=start
            ):
                reference = get_reference(example)
                generated_answer = clean_answer(generation)
                reference_answer = reference["reference_answer"]
                rows.append(
                    {
                        "model_id": model_id,
                        "example_index": index,
                        **reference,
                        "generated_answer": generated_answer,
                        "token_f1": token_f1(generated_answer, reference_answer),
                    }
                )
        predictions = pd.DataFrame(rows)
        reference_scores, reference_logprob_seconds = score_reference_answers(
            model, tokenizer, eval_dataset
        )
        predictions = predictions.merge(
            reference_scores, on="example_index", how="left"
        )
    finally:
        memory_stats = cuda_peak_memory_mb()

    total_generation_seconds = sum(s["inference_seconds"] for s in batch_stats)
    total_generation_tokens = sum(s["generated_tokens"] for s in batch_stats)
    total_inference_seconds = total_generation_seconds + reference_logprob_seconds
    telemetry = {
        "total_inference_seconds": total_inference_seconds,
        "generation_inference_seconds": total_generation_seconds,
        "generation_tokens": total_generation_tokens,
        "generation_tokens_per_second": total_generation_tokens
        / total_generation_seconds
        if total_generation_seconds > 0
        else float("nan"),
        "reference_logprob_inference_seconds": reference_logprob_seconds,
        "reference_logprob_examples_per_second": len(eval_dataset)
        / reference_logprob_seconds
        if reference_logprob_seconds > 0
        else float("nan"),
        **memory_stats,
    }
    return predictions, telemetry


def add_answer_similarity(predictions, embedder):
    predictions = predictions.copy()
    predictions["answer_similarity"] = float("nan")
    valid = (
        predictions["generated_answer"].notna()
        & predictions["reference_answer"].notna()
        & predictions["generated_answer"].str.len().gt(0)
        & predictions["reference_answer"].str.len().gt(0)
    )
    if valid.any():
        generated = embedder.encode(
            predictions.loc[valid, "generated_answer"].tolist(),
            normalize_embeddings=True,
        )
        reference = embedder.encode(
            predictions.loc[valid, "reference_answer"].tolist(),
            normalize_embeddings=True,
        )
        predictions.loc[valid, "answer_similarity"] = (generated * reference).sum(
            axis=1
        )
    return predictions


def summarize(predictions, telemetry):
    return {
        "model_id": predictions["model_id"].iat[0],
        "n": len(predictions),
        "token_f1_mean": predictions["token_f1"].mean(),
        "token_f1_std": predictions["token_f1"].std(),
        "answer_similarity_mean": predictions["answer_similarity"].mean(),
        "answer_similarity_std": predictions["answer_similarity"].std(),
        "reference_answer_nll_mean": predictions["reference_answer_nll_mean"].mean(),
        "reference_answer_nll_std": predictions["reference_answer_nll_mean"].std(),
        "reference_answer_token_perplexity_mean": predictions[
            "reference_answer_token_perplexity"
        ].mean(),
        "reference_answer_token_perplexity_std": predictions[
            "reference_answer_token_perplexity"
        ].std(),
        "reference_answer_token_perplexity_median": predictions[
            "reference_answer_token_perplexity"
        ].median(),
        "reference_answer_tokens_mean": predictions["reference_answer_tokens"].mean(),
        "reference_answer_tokens_std": predictions["reference_answer_tokens"].std(),
        **telemetry,
    }


In [ ]:
embedder = None

predictions, telemetry = generate_model_predictions(MODEL_NAME, dataset_small["test"])
if embedder is None:
    embedder = SentenceTransformer(EMBED_MODEL_ID, device="cpu")
predictions = add_answer_similarity(predictions, embedder)
metrics = summarize(predictions, telemetry)


In [ ]:
run_metrics = {
    "run_id": pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S"),
    "base_model": MODEL_NAME,
    "new_model_name": NEW_MODEL_NAME,
    "seed": SEED,
    "configured_train_size": TRAIN_SIZE,
    "configured_validation_size": VALIDATION_SIZE,
    "configured_test_size": TEST_SIZE,
    "train_size": len(dataset_small["train"]),
    "validation_size": len(dataset_small["evaluation"]),
    "test_size": len(dataset_small["test"]),
    "max_new_tokens": MAX_NEW_TOKENS,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "lora_bias": lora_config.bias,
    "lora_target_modules": ",".join(lora_config.target_modules)
    if isinstance(lora_config.target_modules, (list, tuple, set))
    else lora_config.target_modules,
    "learning_rate": args.learning_rate,
    "num_train_epochs": args.num_train_epochs,
    "per_device_train_batch_size": args.per_device_train_batch_size,
    "per_device_eval_batch_size": args.per_device_eval_batch_size,
    "gradient_accumulation_steps": args.gradient_accumulation_steps,
    "effective_train_batch_size": args.per_device_train_batch_size
    * args.gradient_accumulation_steps,
    "warmup_steps": args.warmup_steps,
    "weight_decay": args.weight_decay,
    "lr_scheduler_type": args.lr_scheduler_type,
    "optim": args.optim,
    "packing": args.packing,
    "completion_only_loss": args.completion_only_loss,
    "bf16": args.bf16,
    "fp16": args.fp16,
    "eval_strategy": getattr(
        args, "eval_strategy", getattr(args, "evaluation_strategy", None)
    ),
    "eval_steps": args.eval_steps,
    "save_steps": args.save_steps,
    "logging_steps": args.logging_steps,
    "load_best_model_at_end": args.load_best_model_at_end,
    **metrics,
}

metrics_output_path = OUTPUT_DIR / "metrics_history.csv"
current_run = pd.DataFrame([run_metrics])
if metrics_output_path.exists():
    previous_runs = pd.read_csv(metrics_output_path)
    metrics_history = pd.concat(
        [previous_runs, current_run], ignore_index=True, sort=False
    )
else:
    metrics_history = current_run

metrics_history.to_csv(metrics_output_path, index=False)
display(metrics_history)
